In [17]:
from crewai import Agent,Task,Crew,Process,LLM
from dotenv import load_dotenv
import agentops
import os 
from pydantic import BaseModel, Field
from typing import List

In [18]:
load_dotenv()

llm = LLM(
    #take name of model and api_key , temprature
    model="openai/gpt-5.6-luna",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)


In [19]:
session = agentops.init(
    api_key=os.getenv("AGENTOPS_API_KEY"),
    #this code to prevent close after first agent 
    skip_auto_end_session=True
)
print(session)

None


In [20]:
outpur_dir = "./ai-agent-output"
os.makedirs(outpur_dir,exist_ok=True)

In [21]:
no_keywords = 10
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine",
                               min_items = 1, max_items = no_keywords)

#every agent can make one task or multiple tasks
#each agent has role, goal, backstory,llm  
Search_Quaries_Recommendition_Agent = Agent(
    role="Search_Quaries_Recommendition_Agent",
    goal="\n".join([
        "to provide alist of suggested search queries to be passed to the search engine",
        "the queries must be varied and looking for specific items"
           ]),
    backstory="You are an expert search query strategist specializing in product research and online shopping. You analyze user requests and transform them into clear, specific, and diverse search queries. Your goal is to cover different brands, models, specifications, price ranges, and purchasing options to help search engines find the most relevant products and deals."    ,
    llm = llm,
    verbos=True
)
#each task has description , expected output and agent to do this task , expected output , output json
# optional can take async,output file 
Search_Quaries_Recommendition_Task = Task(
    description="\n".join([
        "Generate {no_keyword} highly relevant and diverse search queries for finding {product_name}.",

        "The product must be available for purchase and deliverable in {country_name}.",

        "Search only within the following e-commerce websites:",
        "{website_list}",

        "Do not search for or include blog posts, articles, reviews, forums, news websites, or informational pages.",

        "Focus exclusively on product listing pages, product pages, online stores, and e-commerce marketplaces.",

        "Each search query should be specific and target actual purchasable products.",

        "Vary the queries by considering different brands, models, specifications, features, price ranges, and product variations.",

        "Make sure the generated queries are suitable for use directly in a search engine.",

        "Return exactly {no_keyword} search queries and do not provide any additional explanation."
    ]),
    #json to prevent words in introduction and conclusion
    expected_output="A JSON containing alist of suggested search queries",
    #we will create pydantic scheme for output json 
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(outpur_dir,"step1.json"),
    agent=Search_Quaries_Recommendition_Agent
)

C:\Users\PC\AppData\Local\Temp\ipykernel_23052\4203421257.py:3: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine",
C:\Users\PC\AppData\Local\Temp\ipykernel_23052\4203421257.py:3: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine",


In [22]:
run = Crew(
    agents=[Search_Quaries_Recommendition_Agent],
    tasks=[Search_Quaries_Recommendition_Task],
    process=Process.sequential
)

In [26]:
results =  await run.kickoff_async(
    inputs={
        "product_name": "coffee machine for the office",
        "website_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keyword": 10,
    }
    )

ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 
'flow_started' (expected 'llm_call_started')


ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
🖇 AgentOps: Error in span Search_Quaries_Recommendition_Agent.agent: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
🖇 AgentOps: Error in span Generate 10 highly relevant and diverse search queries for finding coffee machine for the office.
The product must be available for purchase and deliverable in Egypt.
Search only within the following e-commerce websites:
['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']
Do not search for or include blog po

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🖇 AgentOps: Error in span crewai.workflow: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}